# [SQL 재현] 2023년 의료기관별 시군구별 진료비 분석

## 단계: 01. 데이터 전처리 — SQL 재현
- 목표: PY_01에서 pandas로 진행한 전처리 과정을 SQL 쿼리로 다시 작성하고, 결과가 PY_01과 같은지 대조한다.
- 환경: Jupyter Notebook + sqlite3 (파이썬에 기본 내장된 파일형 데이터베이스)
- 대조 기준: PY_01_Preprocessing.ipynb 실행 결과

### 1.1 환경 설정 및 데이터 로드
#### 1.1-1 CSV 불러오기 및 컬럼명 정리
- 공공데이터포털 CSV를 pandas로 불러온다 (cp949 인코딩).
- 괄호가 들어간 컬럼명 2개는 SQL에서 쓸 때마다 따옴표가 필요하므로 짧게 바꾼다 (SAS v2와 같은 이름).

In [2]:
import sqlite3
import pandas as pd

df = pd.read_csv(r'C:\data\hira_sigungu_2023.csv', encoding='cp949')
df = df.rename(columns={
    '보험자부담금(선별포함)' : '보험자부담금',
    '요양급여비용총액(선별포함)' : '요양급여비용총액'
})

#### 1.1-2 SQLite 데이터베이스에 테이블 저장
- DB 파일(sql_practice.db)에 연결하고, DataFrame을 `hira` 테이블로 저장한다.
- 출력되는 숫자는 저장된 행 수.

In [3]:
conn = sqlite3.connect(r'C:\data\sql_practice.db')
df.to_sql('hira', conn, if_exists='replace', index=False)

251

### 1.2 데이터 구조 및 타입 파악
#### 1.2-1 상위 3행 조회
- pandas `df.head(3)` 대응: `SELECT *` + `LIMIT 3`

In [4]:
q = """
SELECT *
FROM hira
LIMIT 3
"""
pd.read_sql(q, conn)

,진료년도,시도,시군구,환자수,명세서청구건수,입내원일수,보험자부담금,요양급여비용총액
0,2023,서울,강남구,3182688,18499074,20105055,2302379610700,2962352500080
1,2023,서울,강동구,1060832,10934670,12123283,835458119590,1108786307310
2,2023,서울,강서구,1163258,11007256,11812064,699996239970,946311204440


#### 1.2-2 행 수 및 그룹 수 확인
- pandas `df.shape[0]`, `df['시도'].nunique()` 대응: `COUNT(*)`, `COUNT(DISTINCT 열)`
- SAS 70~76행 PROC SQL을 SQLite로 옮긴 것

In [5]:
q = """
SELECT COUNT(*) AS 행수,
       COUNT(DISTINCT 시도) AS 시도수,
       COUNT(DISTINCT 시군구) AS 시군구수
FROM hira
"""
pd.read_sql(q, conn)

,행수,시도수,시군구수
0,251,17,250


#### 1.2-3 시도별 시군구 수
- pandas `df['시도'].unique()` 확인 + 시도별 개수: `GROUP BY` + `COUNT(*)`

In [7]:
q = """
SELECT 시도,
       COUNT(*) AS 시군구수
FROM hira
GROUP BY 시도
ORDER BY 시군구수 DESC
"""
pd.read_sql(q, conn)

,시도,시군구수
0,경기,42
1,서울,25
2,경북,24
3,전남,22
4,경남,22
5,강원,18
6,충남,16
7,부산,16
8,전북,15
9,충북,14


#### 1.2-4 컬럼 타입 확인
- pandas `df.info()` 대응: `PRAGMA table_info(테이블명)`
- PRAGMA는 SQLite 전용 명령 (MySQL에서는 다른 명령을 씀)

In [8]:
pd.read_sql("PRAGMA table_info(hira)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,진료년도,INTEGER,0,None,0
1,1,시도,TEXT,0,None,0
2,2,시군구,TEXT,0,None,0
3,3,환자수,INTEGER,0,None,0
4,4,명세서청구건수,INTEGER,0,None,0
5,5,입내원일수,INTEGER,0,None,0
6,6,보험자부담금,INTEGER,0,None,0
7,7,요양급여비용총액,INTEGER,0,None,0


> **행 251개, 열 8개, 시도 17개 — PY_01과 동일** <br>
> `to_sql`로 251행이 모두 저장됐고, 상위 3행(강남구·강동구·강서구) 값이 PY_01 `df.head(3)`와 일치. <br>
> 시도별 시군구 수(경기 42, 서울 25, 경북 24 …)가 PY_02 `sido_agg`와 일치하고 합계 251. <br>
> 컬럼 타입: 정수(INTEGER) 6개, 문자열(TEXT) 2개 — PY_01 `info()`의 int64 6개, object 2개와 일치.

> **시군구 이름은 250종류 — 행 수(251)보다 1개 적음** <br>
> 같은 이름을 가진 시군구가 있다는 뜻. 1.3-5에서 강원 고성군·경남 고성군으로 확인.

### 1.3 데이터 정제 (결측치, 중복값, 공백 확인)
#### 1.3-1 결측치 확인
- pandas `df.isnull().sum()` 대응
- `COUNT(*)`(전체 행 수) − `COUNT(열)`(그 열에서 값이 있는 행 수) = 결측 수

In [9]:
q = """
SELECT COUNT(*) - COUNT(진료년도) AS 진료년도,
       COUNT(*) - COUNT(시도) AS 시도,
       COUNT(*) - COUNT(시군구) AS 시군구,
       COUNT(*) - COUNT(환자수) AS 환자수,
       COUNT(*) - COUNT(명세서청구건수) AS 명세서청구건수,
       COUNT(*) - COUNT(입내원일수) AS 입내원일수,
       COUNT(*) - COUNT(보험자부담금) AS 보험자부담금,
       COUNT(*) - COUNT(요양급여비용총액) AS 요양급여비용총액
FROM hira
"""
pd.read_sql(q, conn)

,진료년도,시도,시군구,환자수,명세서청구건수,입내원일수,보험자부담금,요양급여비용총액
0,0,0,0,0,0,0,0,0


#### 1.3-2 중복 확인 ① 중복된 시도-시군구 조합 찾기
- pandas `df[['시도','시군구']].duplicated()` 대응
- `GROUP BY`로 시도-시군구 조합별로 묶은 뒤, `HAVING`으로 2번 이상 나온 조합만 남긴다

In [14]:
q = """
SELECT 시도, 시군구, COUNT(*) AS cnt
FROM hira
GROUP BY 시도, 시군구
HAVING cnt > 1
"""
pd.read_sql(q, conn)

,시도,시군구,cnt


#### 1.3-3 중복 확인 ② 중복 조합 개수 세기
- 1.3-2 쿼리를 괄호로 감싸 서브쿼리(쿼리 안의 쿼리)로 쓰고, 바깥에서 행 수를 센다
- SAS 96~105행 PROC SQL을 SQLite로 옮긴 것

In [17]:
q = """
SELECT COUNT(*) AS 중복_조합수
FROM (
    SELECT 시도, 시군구, COUNT(*) AS cnt
    FROM hira
    GROUP BY 시도, 시군구
    HAVING cnt > 1
)
"""
pd.read_sql(q, conn)

,중복_조합수
0,0


#### 1.3-4 공백 확인
- PY_01은 `환자수`에 공백이 있는지 확인했다. 이번에는 문자열 열인 `시군구`도 함께 확인한다.
- `LIKE '% %'`: 앞·중간·뒤 어디든 공백이 들어 있는 값

In [18]:
q = """
SELECT SUM(CASE WHEN CAST(환자수 AS TEXT) LIKE '% %' THEN 1 ELSE 0 END) AS 환자수_공백,
       SUM(CASE WHEN 시군구 LIKE '% %' THEN 1  ELSE 0 END)
FROM hira
"""
pd.read_sql(q, conn)

,환자수_공백,SUM(CASE WHEN 시군구 LIKE '% %' THEN 1 ELSE 0 END)
0,0,0


#### 1.3-5 시군구 이름 단독 중복 확인
- 1.2-2에서 행은 251개인데 시군구 이름은 250종류 → 같은 이름이 서로 다른 시도에 있다는 뜻 (1.3-3에서 시도+시군구 조합 중복은 0)
- WHERE 안에 서브쿼리를 넣어 겹치는 이름을 찾고, 어느 시도인지 함께 조회한다

In [23]:
q = """
SELECT 시도, 시군구
FROM hira
WHERE 시군구 IN (
    SELECT 시군구
    FROM hira
    GROUP BY 시군구
    HAVING COUNT(*) > 1
)
"""
pd.read_sql(q, conn)

,시도,시군구
0,강원,고성군
1,경남,고성군


> **결측치 0, 시도+시군구 중복 0, 공백 0 — PY_01과 동일** <br>
> `COUNT(*) - COUNT(열)`로 8개 열 모두 결측 0. <br>
> 시도+시군구 조합으로 묶었을 때 2번 이상 나온 조합 없음 (HAVING 결과 빈 표, 서브쿼리로 센 개수 0). <br>
> 환자수뿐 아니라 문자열 열인 시군구에도 공백 없음.

> **시군구 이름 단독 기준으로는 1건 겹침: 고성군 (강원·경남)** <br>
> 시도가 다른 별개 지역이므로 오류가 아님. PY_01은 시도+시군구 조합으로만 중복을 확인해 드러나지 않았고, SQL로 옮기며 `COUNT(DISTINCT 시군구)`를 추가로 세다가 발견했다. <br>
> → 이후 조인·필터·중복 확인은 시군구 단독이 아니라 시도+시군구를 함께 기준으로 쓴다.

### 1.4 변수 생성 및 단위 변환
#### 1.4-1 원 → 억원 변환 (나눗셈 방식 비교)
- pandas `(df['요양급여비용총액(선별포함)'] / 1e8).round(1)` 대응
- 나누는 수를 `100000000`(정수)으로 쓸 때와 `100000000.0`(소수점 포함)으로 쓸 때 결과를 나란히 비교한다

In [19]:
q = """
SELECT 시도, 시군구,
       요양급여비용총액/  100000000 AS 억원_A,
       요양급여비용총액 / 100000000.0 AS 억원_B,
       ROUND(요양급여비용총액 / 100000000.0, 1) AS 억원_반올림
FROM hira
LIMIT 5
"""
pd.read_sql(q, conn)

,시도,시군구,억원_A,억원_B,억원_반올림
0,서울,강남구,29623,29623.525001,29623.5
1,서울,강동구,11087,11087.863073,11087.9
2,서울,강서구,9463,9463.112044,9463.1
3,서울,관악구,4603,4603.336186,4603.3
4,서울,구로구,8891,8891.151666,8891.2


> **SQLite에서 정수 ÷ 정수는 소수점 아래를 버린다** <br>
> 강남구 진료비를 `100000000`으로 나누면 29623, `100000000.0`으로 나누면 29623.525001. 강남구(29623.525 → 29623), 강동구(11087.86 → 11087)처럼 반올림이 아닌 버림이다. <br>
> 나누는 수에 `.0`을 붙여 소수 계산으로 바꾸고 `ROUND(…, 1)`을 적용하면 29623.5로, PY_01 최대값과 일치한다. <br>
> pandas에서는 정수 열끼리 나눠도 소수까지 계산됐다 (PY_02의 1인당 진료비: 화순군 280.6). → SQL_02에서 1인당 진료비 계산 시 주의할 지점.

### 1.5 기초 통계량
#### 1.5-1 진료비(억원) 요약 통계
- PY_01 `describe()` 중 개수·평균·최소·최대 대응
- 억원 변환은 서브쿼리 안에서 하고, 바깥에서 요약한다 (PY_01과 같은 순서: 반올림 → 요약)

In [20]:
q = """
SELECT COUNT(*) AS 개수,
       AVG(억원) AS 평균,
       MIN(억원) AS 최소,
       MAX(억원) AS 최대
FROM (
     SELECT ROUND(요양급여비용총액 / 100000000.0, 1) AS 억원
     FROM hira
)
"""
pd.read_sql(q, conn)

,개수,평균,최소,최대
0,251,3496.406773,31.4,29623.5


#### 1.5-2 환자수·입내원일수 요약 통계
- PY_01 1.5 `describe()` 중 평균·최소·최대 대응

In [22]:
q = """
SELECT AVG(환자수) AS 환자수_평균,
       MIN(환자수) AS 환자수_최소,
       MAX(환자수) AS 환자수_최대,
       AVG(입내원일수) AS 입내원일수_평균,
       MIN(입내원일수) AS 입내원일수_최소,
       MAX(입내원일수) AS 입내원일수_최대
FROM hira
"""
pd.read_sql(q, conn)

,환자수_평균,환자수_최소,환자수_최대,입내원일수_평균,입내원일수_최소,입내원일수_최대
0,407857.091633,9264,3182688,4.320838e+06,63607,20105055


> **개수·평균·최소·최대 모두 PY_01 `describe()`와 일치** <br>
> 진료비(억원): 251개 / 평균 3,496.406773억 / 최소 31.4억 / 최대 29,623.5억 <br>
> 환자수: 평균 407,857.09명 / 최소 9,264명 / 최대 3,182,688명 <br>
> 입내원일수: 평균 4.320838e+06일 / 최소 63,607일 / 최대 20,105,055일 <br>
> PY_01은 억원으로 반올림한 값을 요약했으므로, SQL에서도 서브쿼리 안에서 반올림한 뒤 바깥에서 요약해 계산 순서를 맞췄다. <br>
> 표준편차·사분위수는 SQL_03에서 윈도우 함수로 확인 예정.